# Домашнее задание № 1

## Побудем занудами, но введем все необходимые формулировки и обозначения

### Neighboring datasets (смежные выборки)

Пусть $\mathcal Z$ — пространство записей, а $D=(z_1,\ldots,z_n)\in\mathcal Z^n$ — выборка из $n$ записей. Выборки $D$ и $D'$ называются **смежными**, и мы пишем $D\sim D'$, если они имеют одинаковый объём и различаются **ровно одной записью**:

$$\exists i\in\{1,\ldots,n\}:\quad z_i\ne z_i',\qquad z_j=z_j'\quad\forall j\ne i.$$

Эта модель называется **replace-one adjacency (смежность с заменой одной записи)**. В дальнейшем я буду использовать  термин «смежные выборки», всегда имея в виду именно это отношение, если прямо не оговорено иное.

### Определение дифференциальной приватности

Рандомизированный механизм $\mathsf M$ удовлетворяет **$(\varepsilon,\delta)$-дифференциальной приватности**, если для любых смежных выборок $D\sim D'$ и любого измеримого множества возможных выходов $S$ выполняется

$$
\Pr\{\mathsf M(D)\in S\}\le e^\varepsilon\Pr\{\mathsf M(D')\in S\}+\delta.
$$

Здесь $\varepsilon\ge0$, $0\le\delta<1$. Вероятности берутся по случайности самого механизма при фиксированных входных выборках. Поскольку $D\sim D'$ — симметричное отношение, определение требует также обратное неравенство с переставленными $D,D'$.

Также введём термин **global sensitivity (глобальную чувствительность)** запроса. Для детерминированного запроса $f:\mathcal Z^n\to\mathbb R^d$ его глобальная чувствительность по евклидовой норме равна

$$\Delta_2(f)=\sup_{D\sim D'}\|f(D)-f(D')\|_2.$$

Напомню, что применение произвольного, в том числе рандомизированного, преобразования к уже приватному выходу без дополнительного обращения к исходным данным сохраняет гарантию DP. 

### Используемые теоремы

**Gaussian mechanism (гауссовский механизм): достаточное условие приватности.** При $0<\varepsilon\le1$, $0<\delta<1$ механизм $f(D)+Z$, где $Z\sim\mathcal N(0,\sigma^2 I_d)$, является $(\varepsilon,\delta)$-DP, если

$$
\sigma\ge\kappa(\varepsilon,\delta)\Delta_2(f),\qquad
\kappa(\varepsilon,\delta)=\frac{\sqrt{2\log(1.25/\delta)}}{\varepsilon}.
$$

На теоретических лекций обсуждалось, что данное условие **достаточное, но не необходимое**. Для положительно определённой ковариационной матрицы $\Sigma$ используется **whitening**, то есть обратимое преобразование $\Sigma^{-1/2}$, приводящее ковариацию к $I_d$. Можно применить это преобразование к выходу, воспользоваться указанной теоремой и затем обратить преобразование.

**Advanced composition (усиленная композиционная теорема).** Если каждый из $T$ механизмов при любой фиксированной истории предыдущих ответов удовлетворяет $(\varepsilon_0,\delta_0)$-DP, их совместный протокол удовлетворяет $(\varepsilon_T,T\delta_0+\delta')$-DP при любом $\delta'>0$, где

$$
\varepsilon_T=\varepsilon_0\sqrt{2T\log(1/\delta')}+T\varepsilon_0(e^{\varepsilon_0}-1).
$$

**Оговорка об обозначениях.** В литературе сокращение GDP может означать отдельное понятие *Gaussian Differential Privacy*. Здесь мы используем только достаточную теорему для гауссовского механизма и не отождествляем её с GDP.

### Обозначения

| Обозначение | Значение |
|:---|:---|
| $D,D'$ | смежные выборки, $D\sim D'$ |
| $n$ | объём исходной выборки |
| $d$ | размерность выходного вектора или параметров модели |
| $\varepsilon,\delta$ | параметры гарантии дифференциальной приватности |
| $\mathsf M$ | рандомизированный механизм |
| $\Sigma$, $\Lambda=\Sigma^{-1}$ | ковариационная матрица шума и precision matrix |
| $P,Q$; $p,q$ | распределения выходов и их плотности относительно общей меры |
| $\theta$, $\mathcal X$ | параметр модели и *выпуклое* допустимое множество |
| $T$, $\gamma$, $C$ | число шагов, размер шага и порог clipping-a |

# Задача 1. Оптимальный гауссовский механизм для медицинской статистики (20 баллов)

**Постановка задачи.** Неинтересно решать абстрактные теоретические задачи, поэтому рассмотрим вот такую постановку. Пусть некоторая больница публикует сведения о количестве пациентов с определённым диагнозом в *двух отделениях* $A$ и $B$. Понимание о том, к какому отделению относится каждый из пациентов считается общедоступным. А вот его диагноз $x_i\in\{0,1\}$ является защищаемой информацией. Предположим, что в каждом отделении имеется хотя бы один пациент. При переходе к смежной выборке **диагноз** одного пациента меняется, но при этом отделение остаётся прежним.

Пусть $N_A(D)$ и $N_B(D)$ обозначают числа пациентов с диагнозом в отделениях. Больнице требуется опубликовать два агрегированных показателя (дополнительный балл за креативное описание того, что эти параметры могут означать):

$$U(D)=N_A(D)+N_B(D),\qquad H(D)=N_A(D)-N_B(D).$$

Больница не промах, глав. врач не дурак, поэтому он применяет рандомизированный механизм

$$
\mathsf M(D)=f(D)+Z,\qquad
f(D)=\begin{pmatrix}U(D)\\ H(D)\end{pmatrix},\quad
Z\sim\mathcal N(0,\Sigma),\quad\Sigma\succ0.
$$

Ковариационная матрица $\Sigma\in\mathbb R^{2\times2}$ выбирается с условием достижения нужной гарантии дифференциальной приватности. Для заданных весов $a,b>0$ ошибка (риск) оценивается `weighted mean squared error`:

$$\mathcal R(\Sigma)=a\mathbb EZ_1^2+b\mathbb EZ_2^2=a\Sigma_{11}+b\Sigma_{22}.$$

Параметры приватности фиксированы: $0<\varepsilon\le1$, $0<\delta<1$. В рамках этой задачи рассматриваются только те ковариационные матрицы, которые бы удовлетворяли GDP.

**а) (3 балла).** Вычислите все возможные векторы $f(D)-f(D')$ для $D\sim D'$. Найдите глобальную чувствительность $\Delta_2(f)$.

**б) (5 баллов).** Для $P=\mathcal N(\mu,\Sigma)$ и $Q=\mathcal N(\mu',\Sigma)$ определите функцию потерь как

$$ L(y) = \log \left( \frac{p(y)}{q(y)} \right) $$ 

и докажите, что $L(Y)$ при $Y\sim P$ является *нормальной случайной величиной*. Также найдите её параметры через $v^\top\Sigma^{-1}v$, $v=\mu- \mu'$ (нужно также отдельно учесть $Y\sim Q$).

**в) (4 балла).** Положив $\Lambda=\Sigma^{-1}$, выразите *достаточное* условие приватности как систему неравенств на элементы $\Lambda$.

**г) (6 баллов).** Найдите глобальный минимум $\mathcal R(\Sigma)$ на полученной допустимой области. Предлагается такая последовательность действий:
- (i) доказать, что у матрицы точности внедиагональные элементы не улучшают целевую функцию,
- (ii) проверить достижимость нижней границы.

**д) Следствия (2 балла).** Укажите точное условие, при котором оптимален isotropic noise (изотропный шум, $\Sigma_{11} = \Sigma_{22}$), и сделайте вывод о применимости.

# Задача 2. Приватное обучение модели страхового риска (25 баллов)

**Постановка задачи.** Страховая компания обучает модель оценки риска (а кто не рискует, тот не пьет шампанское). Отдельные редкие страховые случаи могут создавать большие по норме градиенты. TechLead использует при обучении per-example gradients: каждый вычисляется по одному элементу выборки и для защиты данных клиентов (или мешков с деньгами) выполняет gradient clipping для каждого примера из выборки, а в финале еще и прибавляет гауссовский шум. Нужно установить, какую цену платит метод оптимизации за это ограничение, ведь расплата ждет каждого.

Пусть $\mathcal X\subset\mathbb R^d$ - непустое замкнутое выпуклое множество, для которого известна положительная верхняя оценка диаметра $D_{\mathcal X}>0$:

$$\sup_{u,v\in\mathcal X}\|u-v\|_2\le D_{\mathcal X}<\infty.$$

Пусть также задана эмпирическая функция потерь:

$$F_D(\theta)=\frac{1}{n} \sum_{i=1}^n\ell(\theta;z_i),\qquad\theta\in\mathcal X.$$

Будем считать, что для каждого $z_i$ функция $\ell(\cdot;z_i)$ выпукла и дифференцируема; для всех $\theta\in\mathcal X$ выполнено $\|\nabla\ell(\theta;z_i)\|_2\le G$. Напомню, что оператор clipping-a определяется формулой

$$\operatorname{clip}_C(g)=\begin{cases}g,&\|g\|_2\le C,\\ \frac{C}{\|g\|_2}g,&\|g\|_2>C.\end{cases}$$

Мы рассматриваем следующую схему обновления на $t$-ой без Пуассоновского батчирование (то есть по всей выборке без случайной подвыборки) проективного градиентного спуска (PGD):

$$
\begin{aligned}
\hat{g}_t&=\frac{1}{n} \sum_{i=1}^n\operatorname{clip}_C(\nabla\ell(\theta_t;z_i))\\
\xi_t&\sim\mathcal N(0,\sigma^2 I_d)\\
\theta_{t+1}&=\Pi_{\mathcal X}\bigl(\theta_t-\gamma(\hat{g}_t+\xi_t)\bigr)
\end{aligned}
$$

Здесь $\gamma>0$ — размер шага, $\Pi_{\mathcal X}$ - евклидова проекция на множество $\mathcal{X}$.


**Соглашение о начальном состоянии и итоговом результате.** Считаем, что $\theta_0$ фиксировано независимо от защищаемой выборки, число итераций $T\ge1$, $0<\varepsilon\le1$, $0<\delta<1$. Публикуется среднее $\overline\theta_T=T^{-1}\sum_{t=0}^{T-1}\theta_t$; вспомогательный протокол зашумлённых градиентов рассматривается только для доказательства приватности.

**а) (4 балла).** При фиксированном значении параметров $\theta_t$ на $t$-ой итерации алгоритма получите глобальную чувствительность для $\hat{g}_t$ на смежных выборках и сделайте вывод о зависимости или независимости итераций данного метода.

**б) (5 баллов).** Используя только достаточную гауссовскую калибровку и усиленную теорему о композиции, выберите явное $\sigma$, гарантирующее $(\varepsilon,\delta)$-DP. Полезно для сокращения выкладок определить вспомогательный параметр $\varepsilon_0$ как единственный корень указанного уравнения композиции.

**в) (4 балла).** Положим

$$S_C(D)=\sup_{\theta\in\mathcal X}\frac1n\sum_{i=1}^n\bigl(\|\nabla\ell(\theta;z_i)\|_2-C\bigr)_+.$$

Докажите $\|\nabla F_D(\theta)-\hat{g}(\theta)\|_2\le S_C(D)$; дополнительно полезно построить пример выпуклых дифференцируемых потерь, в котором направление среднего градиента меняется на противоположное после ограничения норм.

**г) (8 баллов).** Докажите теорему сходимости предложенного метода, а именно

$$\mathbb E\left[F_D(\overline\theta_T)-F_D(\theta^*)\right]\le
\frac{D_{\mathcal X}^2}{2\gamma T}+\frac\gamma2(C^2+d\sigma^2)+D_{\mathcal X}S_C(D).$$

**д) (4 балла).** Найдите оптимальный шаг для данного алгоритма. В качестве $\sigma$ выберите результат из пункта (б) и объясните влияние числа шагов $T$ и порога $C$ при фиксированных $(\varepsilon,\delta)$.

# Задача 3. Адаптивная публикация городской статистики (20 баллов)

**Постановка задачи.** Городской центр аналитики отвечает на запросы исследователя к приватной выборке. Сначала исследователь получает один результат, затем выбирает следующий запрос на основе уже полученных ответов. Это так называемые **adaptive queries (адаптивные запросы)**. Усложнение состоит в том, что запросы не фиксируются заранее, поэтому выходы последовательных механизмов могут зависеть друг от друга.

Пусть $T\ge1$, $\eta>0$. Сравним работу системы на смежных выборках $D\sim D'$. Через $Y_t$ обозначим ответ в момент $t$, а через $Y_{<t}=(Y_1,\ldots,Y_{t-1})$ - историю предыдущих ответов. Совместные распределения полного протокола взаимодействия $Y_{1:T}$ обозначим $P$ (для $D$) и $Q$ (для $D'$).

**Предположение об условной приватности.** Выдвинем следующее предположение: для всякой возможной общей истории $y_{<t}$ условные распределения следующего ответа при $D$ и $D'$ удовлетворяют $(\eta,0)$-DP в обе стороны (то есть для $D \sim D'$ и $D' \sim D$). Будем считать, что условные плотности $p_t(\cdot\mid y_{<t})$ и $q_t(\cdot\mid y_{<t})$ существуют относительно общей меры, а их отношения определены почти наверное при обоих законах (это важно, иначе ничего не получится, а вы будете говорить, что так не работает статистика).

**а) (4 балла).** Определим случайную величину $L_T=\log(dP/dQ)(Y_{1:T})$ - потерю приватности полного протокола. Найдите представление $L_T$ с явно записанными условными приращениями $L_t$.

**б) (5 баллов).** Для двух распределений, удовлетворяющих симметричной $\eta$-DP, докажите

$$0\le \mathrm{KL}(P\|Q)\le\eta(e^\eta-1).$$

*Подсказка:* сначала оцените сумму $\mathrm{KL}(P\|Q)+\mathrm{KL}(Q\|P)$.

**в) (6 баллов).** Пользуясь условной леммой Хёффдинга и последовательным взятием условных математических ожиданий, докажите для любого $0<\delta<1$

$$\mathbb{P}_{p \sim P}\left\{L_T>T\eta(e^\eta-1)+\eta\sqrt{2T\log(1/\delta)}\right\}\le\delta.$$

Выведите из этого оценку $(\varepsilon_T,\delta)$-DP для обоих порядков смежных выборок ($D \sim D'$ и $D' \sim D$).

**Напоминание.** Если при фиксированной информации $\mathcal G$ случайная величина $X$ принимает значения в $[a,b]$, то для любого $\lambda\in\mathbb R$

$$\mathbb E\left[e^{\lambda(X-\mathbb E[X\mid\mathcal G])}\mid\mathcal G\right]\le\exp\left(\frac{\lambda^2(b-a)^2}{8}\right).$$

Здесь требуется использовать именно условное неравенство, а не предполагать независимость ответов.

**г) (3 балла).** Покажите, что та же гарантия справедлива, если исследователь прекращает запросы по правилу, зависящему только от уже опубликованных ответов, и при этом делает не более $T$ запросов.

**д) (2 балла).** Можно ли считать приращения $L_t$ независимыми? Если нет, то как условная концентрация решает эту проблему?

# Задача 4. Статистические пределы аудита обученной модели (20 баллов)

**Постановка задачи.** **Privacy auditing** предназначен для проверки заявленных гарантий алгоритма. Рассмотрим независимого аудитора, проверяющего процедуру обучения медицинской модели. Есть подозрение, что в некоторых случаях алгоритм запоминает редкую запись и раскрывает её через выход модели. Аудитор умеет многократно запускать *рандомизированную процедуру* на двух смежных выборках и получает независимые наблюдения выходов. Цель — построить математически обоснованный критерий нарушения DP и понять, когда редкую утечку принципиально трудно обнаружить.

Пусть $P$ и $Q$ — распределения выходов алгоритма на фиксированных смежных выборках $D,D'$. Обозначим $\nu=P+Q$ — общую доминирующую меру, $p=dP/d\nu$ и $q=dQ/d\nu$. Такой выбор допускает **непересекающиеся носители** распределений.

Для заданного $\varepsilon\ge0$ определим **hockey-stick divergence** — специальную меру различимости распределений, характеризующую наибольшее нарушение одностороннего неравенства DP:

$$\mathcal H_\varepsilon(P\|Q)=\sup_{S}\bigl[P(S)-e^\varepsilon Q(S)\bigr],$$

где супремум берётся по всем измеримым множествам. Это максимальная величина нарушения одностороннего DP-неравенства при фиксированном $\varepsilon$. Для $(\varepsilon,\delta)$-DP требуется $\mathcal H_\varepsilon(P\|Q)\le\delta$ **и** $\mathcal H_\varepsilon(Q\|P)\le\delta$ для всех смежных пар.

**а) (5 баллов).** Используйте идею леммы Неймана — Пирсона: оптимальное событие для различения двух законов описывается отношением плотностей. Выведите вариационную формулу для $\mathcal H_\varepsilon(P\|Q)$ и найдите множество $S$, доставляющее условие супремума. Установите связь с **total variation distance** при $\varepsilon=0$.

**б) (4 балла).** Докажите, что произвольная рандомизированная постобработка выхода модели не увеличивает $\mathcal H_\varepsilon$. Объясните, почему аудит, основанный только на предсказаниях модели, может быть слабее аудита полного выхода процедуры обучения. (Вспомните про ядра, желательно марковские)

**в) (5 баллов).** Пусть событие $S$ выбрано до получения независимой проверочной выборки. Аудитор имеет $m$ независимых наблюдений из $P$ и $m$ из $Q$. Для фиксированного $\delta$ выведите нижнюю доверительную границу для $\varepsilon$ с доверительным уровнем не менее $1-\alpha$. Используйте неравенство Хёффдинга и объединение двух неблагоприятных событий.

**г) (6 баллов).** Пусть $R_0$ — некоторое распределение, $R_0(\{\star\})=0$, а

$$Q=R_0,\qquad P=(1-\rho)R_0+\rho\delta_\star,\qquad0<\rho<1,$$

где $\delta_\star$ — мера Дирака в символе $\star$. По $m$ независимым наблюдениям из каждого распределения аудитор должен отличить нулевую гипотезу $P=Q=R_0$ от указанной альтернативы. Вероятность ложноположительного решения не превышает $\alpha$. Найдите необходимое число наблюдений для мощности статистического критерия не менее $1-\beta$ и поясните связь результата с $(\varepsilon,\delta)$-DP.

# Мини-практикум. Приватная классификация гистологических изображений (15 баллов)

**Постановка задачи.** Представим, что мы разработали классификатор тканей для медицинского учреждения. На данных этого учреждения модель показывает приемлемое качество, но после передачи в другую клинику начинает ошибаться. Кроме того, обучающие изображения содержат информацию, которую мы не можем раскрывать. Хотелось бы понять, какую часть ухудшения качества объясняет DP-обучение, а какую — изменение распределения данных между клиниками.

Воспользуемся **PathMNIST** из MedMNIST с задачей классификации **девяти типов тканей** на цветных гистологических изображениях. Обучающая и валидационная части происходят из одного источника, а тестовая выборка - специально собрана из другого медицинского учреждения. Для эксперимента используем фиксированную, заранее случайно выбранную часть обучающей выборки; при желании её объём можно увеличить до полного набора.

Нас интересует не максимальная accuracy сама по себе. Если модель всегда предсказывает наиболее распространённый класс, она может показывать приличную accuracy и при этом быть совершенно бесполезной для остальных типов ткани. Поэтому будем сравнивать **macro-F1**, **balanced accuracy**, полноту по каждому классу, качество вероятностных прогнозов и их изменение на тестовой выборке.

Предлагается выполнить четыре обучения с одинаковой архитектурой и начальной инициализацией:

| Обучение | Порог clipping-а | Целевой бюджет | Для чего нужно |
|:--|:--:|:--:|:--|
| Без DP | — | — | Базовое качество архитектуры |
| DP-A | $C=0.5$ | $\varepsilon=4$ | Сильное ограничение индивидуальных градиентов |
| DP-B | $C=2.0$ | $\varepsilon=4$ | Проверяем влияние $C$ при сопоставимом бюджете |
| DP-C | $C=0.5$ | $\varepsilon=8$ | Проверяем влияние бюджета при фиксированном $C$ |

Во всех DP-прогонах положим $\delta=10^{-5}$. Значение `noise_multiplier` для каждого целевого $\varepsilon$ вычисляет Opacus. В частности, при фиксированной схеме sampling изменение $C$ само по себе не меняет учтённый бюджет, если сохраняется отношение стандартного отклонения добавляемого шума к $C$.

**а) Диагностика данных и модели (3 балла).** Сначала вычислите доли классов на открытых данных и качество константного классификатора. Проверьте, что архитектура совместима с Opacus.

**б) Сравнение методов обучения (4 балла).** Обучите все четыре модели. Постройте кривые macro-F1, balanced accuracy и учтённого $\varepsilon$ по эпохам. Сравните два порога $C$ при одинаковом бюджете и два бюджета при одинаковом $C$. Укажите число итераций, время обучения, реальный расход бюджета и пиковое потребление памяти (чтобы оценить насколько больше ест Opacus).

**в) Проверка обученной модели (5 баллов).** По лучшей валидационной эпохе оцените на тестовой выборке полноту каждого класса, матрицы ошибок и калибровку вероятностей. Найдите классы, на которых DP-обучение приводит к наибольшему изменению полноты. Выясните, может ли модель с хорошей валидационной accuracy оказаться хуже при переносе между медицинскими центрами.

**г) Исследование градиентов и принадлежности к обучающей выборке (3 балла).** На диагностической выборке измерьте, насколько разные $C$ меняют средний градиент. Затем проведите атаку на принадлежность к обучающей выборке, используя значение функции потерь как скоринговую функцию. Объясните, почему успех такой атаки ещё не является нижней доверительной границей для $\varepsilon$ всего алгоритма обучения.

### Подготовка окружения

In [ ]:
%pip install -q "medmnist>=3.0,<4" "opacus==1.5.4" "scikit-learn>=1.3" "matplotlib>=3.7" "pandas>=2.0"

In [ ]:
from dataclasses import dataclass
from copy import deepcopy
from pathlib import Path
import gc
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, roc_auc_score
)

import medmnist
from medmnist import PathMNIST, INFO
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager

@dataclass(frozen=True)
class Config:
    seed: int = 42

    train_size: int = 45000    # до 89996: больше данных == больше времени ?= лучше качество
    epochs: int = 12           # если есть желание, то можно поставить любую произвольную, только напишите
    logical_batch: int = 128   # один логический шаг оптимизатора
    physical_batch: int = 12   

    learning_rate: float = 1.5e-3
    weight_decay: float = 1e-4
    workers: int = 2           
    
    delta: float = 1e-5
    clip_low: float = 0.5
    clip_high: float = 2.0
    epsilon_low: float = 4.0
    epsilon_high: float = 8.0

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'


def set_seed(seed):
    """Воспроизводимость учебного опыта. Не использовать публичный seed для production DP."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
print('Устройство:', DEVICE)
print('Версии:', 'torch', torch.__version__, '| opacus', __import__('opacus').__version__,
      '| medmnist', medmnist.__version__)
if PIN_MEMORY:
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM, ГБ:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    warnings.warn('GPU не обнаружен: полный эксперимент существенно медленнее.')

### а) Сначала посмотрим на задачу, которую мы решаем

Ниже создаются все нужные даталоадеры для обучающей, валидационной и тестовой выборок. Индексы обучающих изображений выбираются независимо от их содержимого и меток. Это важно: если формировать «сбалансированную» приватную выборку с помощью незащищённого анализа её меток, такую дополнительную обработку нельзя автоматически считать бесплатной с точки зрения DP.

In [ ]:
TRAIN_TF = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
EVAL_TF = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Чтение файлов из исходных разбиений MedMNIST
full_train_aug = PathMNIST(split='train', transform=TRAIN_TF, download=True, size=28)
full_train_eval = PathMNIST(split='train', transform=EVAL_TF, download=True, size=28)
val_set = PathMNIST(split='val', transform=EVAL_TF, download=True, size=28)
test_set = PathMNIST(split='test', transform=EVAL_TF, download=True, size=28)

assert len(full_train_aug) == len(full_train_eval)
assert CFG.train_size < len(full_train_aug), 'Оставьте часть train для контрольной non-member выборки.'

train_perm = np.random.default_rng(CFG.seed).permutation(len(full_train_aug))
train_indices = train_perm[:CFG.train_size].tolist()
nonmember_indices = train_perm[CFG.train_size:].tolist()
train_set = Subset(full_train_aug, train_indices)

LABELS = [INFO['pathmnist']['label'][str(i)] for i in range(9)]
NUM_CLASSES = len(LABELS)


def label_counts(dataset, indices=None):
    labels = np.asarray(dataset.labels).reshape(-1)
    if indices is not None:
        labels = labels[np.asarray(indices)]
    return np.bincount(labels, minlength=NUM_CLASSES)

counts = pd.DataFrame({
    'Класс': LABELS,
    'Train (выбранные)': label_counts(full_train_eval, train_indices),
    'Validation': label_counts(val_set),
})
display(counts)

majority = int(np.argmax(label_counts(full_train_eval, train_indices)))
majority_val_acc = label_counts(val_set)[majority] / len(val_set)
print(f'Классификатор, всегда мажоритарный класс {majority} ({LABELS[majority]}):')
print(f'accuracy на validation = {majority_val_acc:.3f}; balanced accuracy = {1 / NUM_CLASSES:.3f}')

fig, ax = plt.subplots(figsize=(11, 4))
counts.set_index('Класс')[['Train (выбранные)', 'Validation']].plot(kind='bar', ax=ax)
ax.set(title='Распределение типов тканей (открытый набор)', ylabel='Число изображений', xlabel='')
ax.grid(axis='y', alpha=0.2)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()

### Архитектура и проверка совместимости

Теперь возьмём небольшую residual CNN с `GroupNorm`. `GroupNorm`, в отличие от `BatchNorm`, не использует статистики других изображений в том же батче. Перед запуском DP-обучения нужно обязательно проверить модель через `ModuleValidator`. Отсутствие замечаний валидатора — необходимая проверка совместимости слоёв.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act = nn.SiLU()
        self.skip = (nn.Identity() if in_channels == out_channels and stride == 1
                     else nn.Conv2d(in_channels, out_channels, 1, stride, bias=False))

    def forward(self, x):
        residual = self.skip(x)
        h = self.act(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act(h + residual)


class HistologyNet(nn.Module):
    def __init__(self, n_classes=9):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, 24), nn.SiLU(),
            ResidualBlock(24, 24),
            ResidualBlock(24, 48, stride=2),
            ResidualBlock(48, 96, stride=2),
            ResidualBlock(96, 96),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.classifier = nn.Linear(96, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x))


def check_architecture(): # выводит количество параметров в сети, а также все несовместимые слои для обучения
    # ваш код
    ...
    return n_params

N_PARAMETERS = check_architecture()

### б) Сопоставим обучение при одинаковой приватности

Для выбора шума используем `PrivacyEngine.make_private_with_epsilon`, Opacus применяет Poisson sampling и `PRV accountant`.

`BatchMemoryManager` разделяет логический мини-пакет на физические части, не добавляя отдельный шум на каждую такую часть. Фиксируем лучшие веса по macro-F1 на валидации.

В реальном приватном обучении начальное состояние генератора шума не должно быть известно атакующему; `secure_mode=False` ниже допустим только для учебного опыта. Кроме того, мы считаем валидационную и тестовую выборки разрешёнными для открытой оценки: иначе публикация метрик по ним тоже потребует анализа приватности.

In [ ]:
def build_train_loader():
    generator = torch.Generator().manual_seed(CFG.seed)
    return DataLoader(train_set, batch_size=CFG.logical_batch, shuffle=True,
                      generator=generator, num_workers=CFG.workers,
                      pin_memory=PIN_MEMORY, drop_last=False)


def build_eval_loader(dataset, batch_size=256):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False,
                      num_workers=CFG.workers, pin_memory=PIN_MEMORY)

VAL_LOADER = build_eval_loader(val_set)
TEST_LOADER = build_eval_loader(test_set)


def expected_calibration_error(y_true, probs, bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for i in range(bins):
        mask = ((conf >= edges[i]) & (conf < edges[i+1])) if i < bins - 1 \
               else ((conf >= edges[i]) & (conf <= edges[i+1]))
        if mask.any():
            ece += mask.mean() * abs((pred[mask] == y_true[mask]).mean() - conf[mask].mean())
    return float(ece)


def evaluate(model, loader):
    # ваш код
    model.eval()
    ...
    return {
        'loss': loss_total / count,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'ece': expected_calibration_error(y_true, probs),
        'recall': recall, 'confusion': cm,
        'labels': y_true, 'probs': probs,
    }


def train_epoch(model, optimizer, loader):
    # ваш код
    model.train()
    seen = 0 # это количество увиденных изображений, функция возвращает его
    ...
    return seen


def unwrap(model):
    return model._module if hasattr(model, '_module') else model


def cpu_weights(model):
    # Сохраняем обычный state_dict без префикса оболочки Opacus.
    return {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}

In [ ]:
EXPERIMENTS = [
    dict(name='Без DP', private=False, C=None, target_epsilon=None),
    dict(name='DP-A: C=0.5, ε=4', private=True,
         C=CFG.clip_low, target_epsilon=CFG.epsilon_low),
    dict(name='DP-B: C=2.0, ε=4', private=True,
         C=CFG.clip_high, target_epsilon=CFG.epsilon_low),
    dict(name='DP-C: C=0.5, ε=8', private=True,
         C=CFG.clip_low, target_epsilon=CFG.epsilon_high),
]


def run_experiment(spec):
    set_seed(CFG.seed)  # контролируем начальную инициализацию
    model = HistologyNet(NUM_CLASSES).to(DEVICE)
    assert not ModuleValidator.validate(model, strict=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate,
                                  weight_decay=CFG.weight_decay)
    loader = build_train_loader()
    engine = None
    sigma = None

    if spec['private']:
        # ваш код
        ...

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    best_f1, best_state, best_epoch = -np.inf, None, None
    history = []
    if PIN_MEMORY:
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    for epoch in range(1, CFG.epochs + 1):
        # ваш код
        ...


    # Не храним сразу четыре модели на GPU.
    del model, optimizer, scheduler, engine, loader
    gc.collect()
    if PIN_MEMORY:
        torch.cuda.empty_cache()
    return result


results = {}
for exp in EXPERIMENTS:
    results[exp['name']] = run_experiment(exp)

summary = pd.DataFrame([
    dict(Модель=name, C=res['C'], Целевой_ε=res['target_epsilon'],
         Реальный_ε=res['final_epsilon'], noise_multiplier=res['noise_multiplier'],
         Лучшая_эпоха=res['best_epoch'], Время_мин=res['seconds']/60,
         Пик_VRAM_ГБ=res['peak_vram_gb'])
    for name, res in results.items()
])
display(summary.round(3))

### Как понять, что модель действительно научилась различать ткани?

Построим историю обучения. Подумайте: если у DP-A и DP-B одинаковый учтённый $\varepsilon$, значит ли это, что они одинаково полезны врачу? Какая часть различий вызвана абсолютным масштабом шума, а какая - смещением градиента при clipping?

In [ ]:
def plot_training(results):
    # ваш код
    ...

plot_training(results)

### в) Проверяем перенос между медицинскими центрами

Теперь один раз оценим выбранные модели на тестовых изображениях другого центра. Кроме общих метрик нужна confusion matrix и полнота по каждому типу ткани. Для медицинского приложения ошибка на редком типе ткани может оказаться важнее небольшого прироста общей accuracy. Рассмотрим также **ECE** - эмпирическую ошибку калибровки уверенности: если модель сообщает вероятность 0.9, насколько часто её предсказания с такой уверенностью действительно правильны?

In [ ]:
def fresh_model(state):
    m = HistologyNet(NUM_CLASSES).to(DEVICE)
    m.load_state_dict(state, strict=True)
    m.eval()
    return m

# ваш код
...

In [ ]:
def reliability_points(y_true, probs, bins=10):
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y_true)
    boundaries = np.linspace(0, 1, bins + 1)
    pairs = []
    for k in range(bins):
        mask = (conf >= boundaries[k]) & (
            conf < boundaries[k+1] if k < bins - 1 else conf <= boundaries[k+1]
        )
        if mask.any():
            pairs.append((float(conf[mask].mean()), float(correct[mask].mean()),
                          int(mask.sum())))
    return pairs

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', label='Идеальная калибровка')
for name, res in results.items():
    test = res['test']
    pairs = reliability_points(test['labels'], test['probs'])
    ax.plot([p[0] for p in pairs], [p[1] for p in pairs], '-o', label=name)
ax.set(xlabel='Средняя уверенность предсказания',
       ylabel='Фактическая доля правильных ответов',
       title='Калибровка на тестовой выборке', xlim=(0, 1), ylim=(0, 1))
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### г) Как clipping меняет направление градиента?

В теоретической задаче 2 мы доказали, что средний clipped gradient может отличаться от исходного. Но насколько это заметно для нашей `HistologyNet`? Возьмём небольшой набор изображений из публичной валидационной части, вычислим индивидуальные градиенты в одной фиксированной точке параметров и сравним их среднее до и после clipping.

Здесь важны две величины: доля градиентов с нормой выше $C$ и угол между средним градиентом до и после ограничения. Первый показатель говорит о частоте изменения векторов, второй - о том, насколько меняется направление предполагаемого шага оптимизации.

In [ ]:
def public_clipping_diagnostic(state, loader, clipping_values, max_items=96):
    # ваш код
    ...

# ваш код
...

### Membership Inference Attack

Предположим, что атакующий знает изображение и правильный тип ткани и может получить вероятности классов от обученной модели. Часто модель даёт меньшую функцию потерь на тех изображениях, которые встречались при оптимизации. Проверим это с помощью простого score-based membership inference attack.

**Протокол.** Для участников класса «был в обучении» возьмём изображения из выбранной обучающей части, а для класса «не был в обучении» - изображения из оставшейся части исходного train, не использованной ни при обучении, ни при выборе лучшей эпохи. Это позволяет избежать очевидного смешения. Используем одинаковое количество изображений в обеих группах, единый детерминированный препроцессинг и AUC как меру способности различать группы.

In [ ]:
def losses_for_subset(state, subset, max_batch=256):
    model = fresh_model(state)
    loader = build_eval_loader(subset, batch_size=max_batch)
    scores = []
    with torch.no_grad():
        for images, labels in loader:
            x = images.to(DEVICE)
            y = labels.reshape(-1).long().to(DEVICE)
            losses = F.cross_entropy(model(x), y, reduction='none')
            scores.append((-losses).cpu().numpy())
    del model
    return np.concatenate(scores)

n_attack = 1200
assert len(nonmember_indices) >= n_attack
member_subset = Subset(full_train_eval, train_indices[:n_attack])
nonmember_subset = Subset(full_train_eval, nonmember_indices[:n_attack])

attack_rows = []
# ваш код
...

### Вопросы к отчёту

1. **Диагностика.** Сопоставьте accuracy с константным классификатором. По каким наблюдаемым признакам можно отличить коллапс к одному классу от настоящего переобучения? Почему одной accuracy недостаточно?
2. **Теорема и оптимизация.** Сравните DP-A и DP-B. У них одинаковый целевой бюджет $\varepsilon$, но различается $C$. Как меняются абсолютный шум на сумму градиентов, смещение при clipping и macro-F1? Сопоставьте ответ с $S_C(D)$ из задачи 2 и объясните, почему выпуклая оценка сходимости напрямую неприменима к нашей residual CNN и AdamW.
3. **Бюджет.** Сравните DP-A и DP-C. Как меняется `noise_multiplier` после калибровки и что происходит с метриками? Отдельно объясните отличие фактически учтённого $\varepsilon$ от целевого.
4. **Медицинская интерпретация.** На каких типах ткани модель допускает наиболее заметное падение recall после DP-обучения? Что будет, если хорошая общая accuracy скрывает провал по одному классу? Почему нельзя автоматически считать все девять ошибок клинически равноценными?
5. **Перенос.** Сравните validation и test. Где сильнее выражен domain shift и как он взаимодействует с DP?
6. **Надёжность вероятностей.** Становятся ли вероятностные предсказания DP-моделей лучше или хуже откалиброваны? Может ли модель с более высоким macro-F1 иметь худший ECE?
7. **Атака.** Сравните AUC membership inference attack для обычной модели и DP-моделей. Что позволяет заключить этот результат об эмпирическом запоминании, а чего он не позволяет заключить о параметрах $(\varepsilon,\delta)$? Чем отличается этот опыт от формального аудита задачи 4?
8. **Граница защиты.** В чём разница между защитой одного изображения и пациента, которому принадлежит несколько изображений? Какие дополнительные условия потребовались бы для patient-level DP?

**Источники и документация:** [MedMNIST](https://medmnist.com/v2), [Opacus PrivacyEngine](https://opacus.ai/api/privacy_engine.html), [Opacus BatchMemoryManager](https://opacus.ai/api/batch_memory_manager.html), [Opacus ModuleValidator](https://opacus.ai/tutorials/guide_to_module_validator).